# K-Fold Cross Validation

Comparing classifiers (logistic regression, SVM, KNN, random forest) on the heart-disease dataset using k-fold CV.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# Import models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Other imports
from pathlib import Path

## Load dataset

Notebook lives at `model-selection/kfold/main.ipynb`, so the project root is two parents up.

In [3]:
root_dir = Path.cwd().parents[1]
dataset_dir = root_dir / "datasets"

heart_data = pd.read_csv(dataset_dir / "heart.csv")
heart_data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## Separate features and labels

In [4]:
X = heart_data.drop(columns="target")
Y = heart_data["target"]
X.shape, Y.shape

((303, 13), (303,))

## Split the data

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=3)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((242, 13), (61, 13), (242,), (61,))

## Comparing the model performances

In [6]:
models = [LogisticRegression(max_iter=1000), SVC(kernel="linear"), KNeighborsClassifier(), RandomForestClassifier()]

In [7]:
def compare_models_train_test():
    for model in models:
        model.fit(X_train, Y_train)
        test_data_prediction = model.predict(X_test)
        accuracy = accuracy_score(Y_test, test_data_prediction)
        print(f"Accuracy score of {model} = {accuracy}")

In [8]:
compare_models_train_test()

Accuracy score of LogisticRegression(max_iter=1000) = 0.7704918032786885
Accuracy score of SVC(kernel='linear') = 0.7704918032786885
Accuracy score of KNeighborsClassifier() = 0.6557377049180327
Accuracy score of RandomForestClassifier() = 0.7540983606557377


## Cross Validation

In [11]:
cv_score_lr = cross_val_score(LogisticRegression(max_iter=1000), X, Y, cv=5)  # cv -> Number of folds. k = 5 subsets
    # Selecting cv = 5 automatically selects stratified kfold
print(f"Results: {cv_score_lr}, Mean: {cv_score_lr.mean()*100}%")

Results: [0.80327869 0.86885246 0.85245902 0.86666667 0.75      ], Mean: 82.82513661202186%


In [12]:
def compare_models_cv(models):
    for model in models:
        cv_score_lr = cross_val_score(model, X, Y, cv=5)
        print(f"Model: {model}, Results: {cv_score_lr}, Mean: {cv_score_lr.mean() * 100}%")
compare_models_cv(models)

Model: LogisticRegression(max_iter=1000), Results: [0.80327869 0.86885246 0.85245902 0.86666667 0.75      ], Mean: 82.82513661202186%
Model: SVC(kernel='linear'), Results: [0.81967213 0.8852459  0.80327869 0.86666667 0.76666667], Mean: 82.83060109289619%
Model: KNeighborsClassifier(), Results: [0.60655738 0.6557377  0.57377049 0.73333333 0.65      ], Mean: 64.38797814207649%
Model: RandomForestClassifier(), Results: [0.83606557 0.90163934 0.83606557 0.83333333 0.76666667], Mean: 83.47540983606558%
